# Lab 1: Verify Your AWS Environment

Welcome to the Neo4j and AWS AgentCore workshop.

You have your own AWS account for this lab. This notebook proves the account can
actually run the workshop, by building the same things the workshop builds and
then deleting them again.

Run each cell in order, from top to bottom. Every check prints `PASS`, `FAIL`, or
`SKIP` with a reason.

**How to run a cell:** click it, then press Shift and Enter together.

## What this notebook does

The workshop's central deployment builds a container for Amazon Bedrock
AgentCore Runtime, pushes it to ECR through CodeBuild, then wires it to a
Gateway, a Lambda, a secret, and a DynamoDB table. Listing permissions does not
prove any of that works, so this notebook does it for real:

| Step | What it builds |
| --- | --- |
| 2b | Nothing. Calls Claude 4 Sonnet and Opus, streaming and not, then requests Nova and Titan embeddings |
| 3 | A Neo4j connection and a throwaway node |
| 5 | Nothing. Confirms AgentCore is visible at all |
| 6 | The IAM execution roles the build and the Runtime need |
| 7 | An ECR repository and the S3 source bundle CodeBuild reads |
| 8 | An ARM64 container, built by CodeBuild and pushed to ECR |
| 9 | An AgentCore Runtime and endpoint, then invokes it |
| 10 | An AgentCore Memory with the workshop's memory strategies |
| 11 | A Lambda, a Gateway, and a target, then calls a tool through it |
| 12 | A Secrets Manager secret and a DynamoDB table |
| 13 | Nothing in AWS. Confirms this machine can install from PyPI |
| 14 | Deletes all of it and confirms the account is empty again |

**This takes 15 to 25 minutes.** Step 8 is the slow one, because it is a real
container build.

**It creates real resources and real cost.** Step 14 deletes everything and
verifies each resource is gone. If you interrupt the notebook partway, run step
14 on its own before closing the lab.

## What this notebook tests

Step 2b calls the two Claude models the workshop's agents run on, because an
agent is nothing but model calls. It also requests 1,024-dimension embeddings
from the Amazon Nova and Titan models.

## Step 0: Install the workshop toolkit

This notebook runs on a small Python package called `workshop_lab`. The cell
below installs it from the public workshop repository on GitHub.

It is installed rather than shipped next to the notebook because a file placed
next to a notebook does not reach your lab workspace. Downloading it over HTTPS
does, and that is what this cell does.

**This cell takes a few seconds and prints nothing while it runs.** Wait for the
version line before running step 1.

**If your instructor tells you a fix has been published, restart the kernel
before re-running this cell.** Python holds on to a package it has already
imported, so re-running the install on its own leaves you on the old code. Use
Kernel then Restart Kernel, then run from step 0 again.

The package brings `boto3`, the AWS library for Python, with it. `boto3` reads
the credentials Vocareum already placed in this environment, so there is no key
to enter and nothing to configure.

The cell also creates `lab`, the object every step below runs through. It holds
one connection to AWS, the running list of results, and the list of deletions
step 14 runs before it sweeps the account.

In [ ]:
# The utility classes this notebook runs on are installed from the public
# workshop repository rather than shipped beside the notebook. Helper files
# placed in /voc/startercode do not reach a Vocareum workspace, measured
# 2026-08-06; installing over HTTPS from GitHub does.
#
# Pin `heads/main` to `tags/<version>` before teaching a class, so every student
# runs the same code even if the repository moves mid-session.
%pip install --quiet "https://github.com/neo4j-partners/neo4j-sec-filings-graphrag-workshop/archive/refs/tags/v0.13.5.zip#subdirectory=staging"

import workshop_lab
from workshop_lab import Harness, Neo4jProbe

# One Harness owns the session, one client per service, every result the
# summary prints, and every delete step 14 runs before it sweeps. Nothing below
# builds a client or catches a ClientError itself.
lab = Harness(prefix="verify")

print(f"workshop_lab {workshop_lab.__version__} loaded from GitHub.")
print(f"Region: {lab.region}")

## Step 1: Confirm your credentials arrived

Vocareum hands this notebook a temporary access key, a secret key, and a session
token. All three have to be present.

This check matters more than it looks. When a lab session has ended, Vocareum
still hands over the three fields, but they are empty. `boto3` treats an empty
value as no value at all, quietly looks elsewhere, and can end up reporting on a
completely different AWS account. Every check below would then pass while
describing somebody else's account.

So this cell stops the notebook if any of the three is empty. A stopped notebook
is the correct outcome there, because the alternative is a confident answer about
the wrong account.

In [ ]:
# Raises rather than warns. A warning would let every cell below run and report
# confidently about whatever account boto3 found instead.
lab.verify_credentials()

## Step 2: Confirm who you are and where you are

This answers two questions. Which AWS account are you working in, and which
region are you in?

The region matters. AgentCore Runtime also runs in `us-west-2`, but the Nova
multimodal embedding model the workshop uses is in `us-east-1` only, so any other
region will fail later even though everything looks fine now.

Write down the account number this prints. If you ever have to report a problem,
that number is the first thing you will be asked for.

In [ ]:
# Confirms the region first, because nothing below is meaningful in the wrong
# one, then names the account and principal. Unlocks lab.names, which every
# later step uses to build resource names.
ACCOUNT_ID = lab.verify_identity()

## Step 2b: Test the Bedrock models

Every agent in the workshop is a Strands agent, and every turn a Strands agent
takes is a Bedrock model call. This check runs immediately after the notebook
has verified your AWS credentials, account, and region, before Neo4j setup or
any AWS resource creation.

This calls the two chat models twice each and requests two embeddings.

**Two models.** `us.anthropic.claude-sonnet-4-6` is the id the workshop's own
labs already carry, and `us.anthropic.claude-opus-4-6-v1` is the newest Opus the
lab accounts are subscribed to. Both are named by inference-profile id, the
one beginning `us.`. The bare `anthropic.` id is refused for every Claude 4
model with a message about on-demand throughput, and that message says
nothing about whether your account may use the model.

**Twice each.** `Converse` needs `bedrock:InvokeModel`. `ConverseStream` needs
`bedrock:InvokeModelWithResponseStream`. Those are two permissions, and a session
has already been measured where one worked and the other did not, for the same
model, in the same account. A Strands agent streams by default, so the streaming
call is the one lab 4 actually depends on.

A failure here has two causes that look alike and are fixed by different people,
so the result line names which one it is. `not entitled to the model` is an
account subscription, which your instructor has to raise with Vocareum.
`IAM or an SCP refused the call` is the lab's own policy. Quote the line when you
report it.

**Two embeddings.** The retrieval labs use
`amazon.nova-2-multimodal-embeddings-v1:0`. This sends it the smallest
generic-index request and requires one 1,024-dimension vector in return. A
successful HTTP response without that vector is not a usable retrieval check.
It also calls `amazon.titan-embed-text-v2:0` with Titan's native text-embedding
request and requires the same vector shape.

This step creates nothing, so step 14 has nothing to delete for it.

In [ ]:
from workshop_lab import ModelAccess

# The chat and embedding model ids, requests, and refusal sorting are in
# workshop_lab.models.
MODELS_READY = ModelAccess(lab).run()

## Step 3: Connect to Neo4j

The workshop's graph modules read one Neo4j Aura instance. Neo4j is not part of
your AWS account, so nothing above this cell tells you anything about it.

**Replace the four values in the next cell with the credentials you were given.**
They are constants here rather than environment variables so there is nothing to
configure outside this notebook. If you leave the placeholders in place, this
step reports `SKIP` and the rest of the notebook still runs.

This writes a node rather than only connecting. Connecting alone would also
succeed with a read-only credential, and the workshop's reservation module
writes.

Once the workshop starts, the demos read these same four values from a `.env`
file at the repository root. Note the name: every demo in the workshop repository
reads `NEO4J_USERNAME`, not `NEO4J_USER`.

In [ ]:
# ----------------------------------------------------------------------------
# REPLACE THESE FOUR VALUES with the Aura credentials you were given.
# ----------------------------------------------------------------------------
NEO4J_URI = "neo4j+s://REPLACE-ME.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = "REPLACE-ME"
NEO4J_DATABASE = "neo4j"
# ----------------------------------------------------------------------------

# The one check in this notebook that does not touch AWS, so it does not go
# through lab.check. Its result reaches the summary in step 15 as an `extra`
# row, where an unmet prerequisite blocks "Environment is ready" in a way an
# ordinary skipped step does not.
neo4j = Neo4jProbe(NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD, NEO4J_DATABASE)

# Assigned rather than left bare: Jupyter would otherwise print the bool under
# the probe's own PASS/FAIL lines, which reads like a second, contradictory
# verdict.
NEO4J_CONFIGURED = neo4j.configured
NEO4J_READY = neo4j.run()

## Step 4: Name the resources, and clear out anything left over

Everything below creates real AWS resources, and their names are not arbitrary.
This cell prints what this run is set to, so a name you find in the AWS console
later is one you can trace back here. Then it deletes anything an earlier run of
this notebook left in the account.

**One prefix.** Every resource this notebook creates carries `verify` in its
name. That is what keeps these apart from anything else in the account, and what
lets a cleanup find leftovers by name if step 14 never gets to run.

**One tag.** Everything is tagged `WorkshopResource` in the same request that
creates it. The cleanup finds resources by that tag or by the naming prefix,
whichever matches, because a handful of things cannot carry a tag: a gateway
target, which the AWS API has no way to tag, and the log groups AgentCore creates
for itself.

**Role names begin with `workshop-`.** Your account is allowed to manage IAM
roles whose names start with `workshop-`, and no others. A role named outside
that prefix is refused, which is why step 6 builds
`workshop-verify-codebuild-role` rather than `verify-codebuild-role`.

**The cleanup runs here as well as at the end.** Step 14 deletes everything this
notebook made, so on a clean run this finds nothing and says so. It matters when
step 14 did not get to run: you closed the tab, the kernel restarted, or a cell
failed and you stopped. AWS keeps those resources, and the second run of a step
that creates a name already in the account fails with a conflict that reads like
a permission problem but is not one. So the account is emptied first.

This is the earliest point where that is safe to do. It needs to know which
account it is in, and step 2 is what established that.

The `lab` object built in step 0 supplies the rest, and three of its behaviours
are worth knowing before you read the output below:

**A record of every check.** Each check files one row, so step 15 can print a
single summary rather than making you scroll back through the whole notebook.
The cleanup in this cell deliberately files nothing: deleting a previous run's
leftovers says nothing about what your account permits.

**Deletion registered at creation time.** The moment a resource exists, its
delete is filed, and step 14 runs them in reverse order. Registering the delete
next to the create is what stops an exception halfway down from leaving
resources behind in your account.

**Waiting rather than assuming.** An AWS call that returns is not an AWS call
that finished. AgentCore leaves a Memory in `UPDATING` for over a minute after
the update returns, leaves a Runtime in `DELETING` for about two minutes, and
refuses the next call while either is true. So the notebook polls for the
settled state instead of trusting the return.


In [ ]:
from workshop_lab import WORKSHOP_TAG_KEY, WORKSHOP_TAG_VALUE, Teardown

# The harness used to be defined here, 181 lines of it. It now lives in
# workshop_lab.harness and was built in step 0, so this cell only reports what
# this run is set to.
print(f"Resources are named with the prefix {lab.prefix!r}.")
print(f"Everything created is tagged {WORKSHOP_TAG_KEY}={WORKSHOP_TAG_VALUE}.")
print(f"Roles will be created as {lab.names.role('<purpose>')}.")
print()

# Same enumeration step 14 runs, with nothing recorded. See
# workshop_lab.teardown for why it enumerates the account rather than trusting a
# list, and why it leaves anything CloudFormation owns alone.
Teardown.sweep_before_start(lab)


## Step 5: Confirm AgentCore is visible

These three list calls are the cheapest thing AgentCore offers, and they gate
everything below. A refusal on any of them means the account cannot see AgentCore
at all, and no amount of building will change that.

They are the three that take no arguments, which is what makes them a usable
gate. The AgentCore list calls that need an identifier are deliberately not here:
with a made-up identifier, a policy that allows the call only on resources you
own is indistinguishable from a policy that denies the call outright, because IAM
returns the same `AccessDeniedException` for both. `ListGatewayTargets` is
measured in step 11 instead, against a gateway that actually exists.

In [ ]:
control = lab.client("bedrock-agentcore-control")

lab.check("agentcore:ListAgentRuntimes", control.list_agent_runtimes)
lab.check("agentcore:ListGateways", control.list_gateways)
lab.check("agentcore:ListMemories", control.list_memories)

# Gates on the group rather than on one call. all_passed reports False when
# nothing matched, so a step that never ran cannot look like a step that passed.
AGENTCORE_VISIBLE = lab.all_passed("agentcore:")

if AGENTCORE_VISIBLE:
    print("\nOK. AgentCore is visible in this account.")
else:
    print("\nAgentCore is not visible in this account, so steps 6 to 12 will fail.")
    print("Stop here and tell your instructor.")

## Step 6: Create the execution roles

Four services need to act on your behalf during the deployment, and each needs
its own role: CodeBuild to push the image, AgentCore Runtime to pull it, Lambda
to write logs, and AgentCore Gateway to invoke the Lambda.

What each role trusts and what each role is allowed to do is in
`workshop_lab.roles`, as four `RoleSpec` entries. The cell below creates them and
reports. Every role is named `workshop-<prefix>-<purpose>`, and the `workshop-`
part is not decoration: the lab's policy allows `iam:PutRolePolicy` only on
`role/workshop-*`, so a role named anything else is created and then denied its
policy.

The Lambda is named with the `hotel-booking-` prefix for the same kind of reason.
The lab's policy scopes CloudWatch Logs writes to `/aws/lambda/hotel-booking-`,
so a differently named function would be authorized to run and then silently
unable to log.

In [ ]:
from workshop_lab import Roles

# The four roles, their trust policies, and their grants now live in
# workshop_lab.roles, where tests hold the names to the `workshop-` prefix
# lab.template grants against. This cell creates them and reports.
roles = Roles(lab)
roles.create_all()

CODEBUILD_ROLE = roles["codebuild-role"]
RUNTIME_ROLE = roles["runtime-role"]
LAMBDA_ROLE = roles["lambda-role"]
GATEWAY_ROLE = roles["gateway-role"]

if roles.all_created:
    print("\nOK. All four execution roles exist and carry their policies.")
else:
    print("\nAt least one role is missing or has no policy. Later steps will skip.")

## Step 7: Create the ECR repository and upload the build source

CodeBuild does not read this notebook. It reads a zip from S3, so the container
source has to exist as files in a bucket before a build can start.

The Dockerfile and the server are in `workshop_lab.build_source`. The bucket name
and the layout are the ones the AgentCore starter toolkit uses, because that is
what the workshop calls. One bucket serves the whole account and region, so a
bucket that already exists is the normal case. Teardown empties it and deletes
it anyway: nothing else in a student account creates it, and objects left in it
by an earlier session are what stops it going.

The Dockerfile deliberately keeps the workshop's own base image,
`ghcr.io/astral-sh/uv:python3.12-bookworm-slim`. That image is not on AWS, so
pulling it is the one part of the build that depends on outbound internet access
from inside CodeBuild. If the lab blocks that egress, step 8 is where you find
out.

The application is the smallest thing an AgentCore Runtime accepts: an HTTP
server answering `GET /ping` and `POST /invocations` on port 8080. It is not an
agent. It exists so the Runtime can start, be invoked, and prove the path works.

In [ ]:
from workshop_lab import BuildSource

# The Dockerfile, the server, and the zip layout are in workshop_lab.build_source.
# This cell creates the ECR repository, makes sure the shared source bucket is
# there, and uploads the archive CodeBuild will be pointed at.
source = BuildSource(lab)
BUILD_SOURCE_READY = source.prepare()

if BUILD_SOURCE_READY:
    print("\nOK. The repository exists and the build source is in S3.")
else:
    print("\nStep 8 has nothing to build. It will report what is missing.")

## Step 8: Build the container with CodeBuild

This is the slow step and the one most likely to fail. Give it 5 to 10 minutes.

AgentCore Runtime accepts ARM64 images only, and this environment has no Docker
daemon, so the build runs in CodeBuild on an ARM container with `privilegedMode`
turned on. Privileged mode is what lets a build run Docker inside a container,
and it is the single setting this whole step depends on.

The build does four things, and each fails for a different reason:

1. `uname -m` prints the build architecture. Anything but `aarch64` means the
   image will not run on AgentCore.
2. `docker login` gets an ECR token using the CodeBuild role.
3. `docker build` pulls the base image from `ghcr.io`. A failure here is network
   egress, not permissions.
4. `docker push` writes the image to your repository.

If the build fails, this prints the tail of the build log, which names which of
the four it was. `workshop_lab.build` holds the buildspec and the reporting.

### The fallback: copying a pre-built image

The build above is expected to work, and as of 2026-08-07 it does. It did not
for most of this workshop's development: every CodeBuild build in a student
account was stopped a few seconds after it started, before any build command
ran. So the workshop also publishes the same image it would have built, and
`workshop_lab.registry` copies that image into your own ECR when the build
produces nothing.

The copy speaks the container registry's HTTP API directly, because this
environment has no Docker daemon, and it has to land in your own private
repository, because AgentCore Runtime refuses to pull an image from anywhere
else.

The fallback is chosen by asking ECR whether the image is there, not by reading
the build status. When the build works, the image is already in the repository,
the copy does not run, and you will see no `registry:` lines below. When it does
not, the copy runs and step 9 gets its image anyway.

In [ ]:
from workshop_lab import ContainerBuild

# The buildspec, the stopped-build detection, the log tail, and the registry
# fallback are in workshop_lab.build. This cell runs them and reports.
container = ContainerBuild(lab, CODEBUILD_ROLE)
IMAGE_PUSHED = container.run()

if IMAGE_PUSHED:
    print(f"\nOK. {lab.names.image_uri} exists and step 9 can pull it.")
else:
    print("\nNo image in ECR. Steps 9 and 10 will skip.")

## Step 9: Create an AgentCore Runtime and invoke it

This is the resource the workshop's deployment produces. It takes the image step
8 pushed, runs it as a managed service, and answers invocations.

The cell creates the Runtime, waits for `READY`, adds an endpoint, waits again,
then sends a payload through the data plane and checks the container echoed it
back. A Runtime that reaches `READY` but returns nothing on invoke means the
container started and then failed its health check, which is a different problem
from a permission refusal.

In [ ]:
from workshop_lab import AgentRuntime

runtime = AgentRuntime(lab, image_pushed=IMAGE_PUSHED, runtime_role=RUNTIME_ROLE)
runtime.run()


## Step 10: Create an AgentCore Memory and tag it

The workshop's memory module creates a Memory with two long-term strategies, then
tags it, then reads the tag back and refuses to continue if it did not stick.

That last part is why this step tags rather than only creating. The workshop's
cleanup deletes only resources carrying the `WorkshopResource` tag. If tagging
silently fails, the workshop stops with a confusing error, and anything already
created stays in the account past the end of the lab.

`UpdateMemory` returns immediately and leaves the Memory in `UPDATING` for over a
minute. Deleting inside that window is refused, so this waits for `ACTIVE` again
before step 14 gets a chance to try.

In [ ]:
from workshop_lab import Memory

memory = Memory(lab)
memory.run()


## Step 11: Build the Gateway boundary

The workshop exposes its one write operation as a Lambda behind an AgentCore
Gateway, so the agent discovers a tool rather than being handed database access.
This step builds that shape: a Lambda, a Gateway, a target joining them, and one
signed request to prove the Gateway answers.

Two details are easy to get wrong and are both handled here. The Gateway target
requires `credentialProviderConfigurations`, which the SDK does not list as
required and the service rejects the request without. And `InvokeGateway` has no
SDK operation at all, so the only way to exercise it is a hand-signed SigV4
request to the Gateway's MCP endpoint.

The last two checks are the ones most easily faked. `tools/list` is answered
from the Gateway's own configuration and never reaches the Lambda, so it passes
on a Gateway that could not invoke anything. The workshop's agent only ever
calls a tool, so this step calls one, and it passes only when the Lambda's own
output comes back. A refused invoke arrives as HTTP 200 with the error inside
the body, which is why the status code alone is not the answer.

The resource policy that permits that call is written by `lambda:AddPermission`,
measured here as its own check. Provisioning runs it and stops when it is
refused, so an account can hold a gateway role that already allows
`lambda:InvokeFunction` and still fail the deployment on this one call.

This is also where `ListGatewayTargets` is measured, against the gateway created
just above. That call is not optional for the workshop: provisioning uses it to
find an existing target by name when it re-runs, and the cleanup module has to
list a gateway's targets before it can delete them. If it is refused here,
gateways cannot be torn down and they survive the end of the lab.

In [ ]:
from workshop_lab import GatewayBoundary

boundary = GatewayBoundary(lab, lambda_role=LAMBDA_ROLE, gateway_role=GATEWAY_ROLE)
boundary.run()

## Step 12: Secrets Manager and DynamoDB

The deployed Lambda reads its Neo4j credentials from Secrets Manager rather than
from an environment variable, and the memory module stores bookings in DynamoDB.
Both are quick.

The secret holds four placeholder strings. Real Aura credentials do not belong in
anything that gets written to a log, and step 3 already proved they work.

The table uses a composite key and on-demand billing, matching the workshop's
table. Tagging leaves a table busy for longer than the item calls take, and
`DeleteTable` refuses a table whose tags are still settling, so the delete
registered here retries rather than assuming.

In [ ]:
from workshop_lab import DataStores

stores = DataStores(lab)
stores.run()

## Step 13: Confirm this machine can install from PyPI

This step creates nothing in AWS. It runs, on this machine, the one thing every
lab in the workshop needs before it can run a single cell.

Each lab installs its own dependencies from PyPI in its first cell:
`neo4j-graphrag[bedrock]` in labs 2 and 3, `strands-agents` in lab 4,
`neo4j-agent-memory` in lab 5, `bedrock-agentcore-starter-toolkit` in both
deploy notebooks, `mcp` and `httpx` in lab 6. If this machine cannot reach PyPI,
every one of those stops at its first cell, and it stops with a pip resolver
error that names a package rather than the network.

Nothing about that is an AWS permission, so every check above can pass while
this fails. That is why it is measured here rather than assumed.

The check asks PyPI for the workshop's own package, `neo4j-graphrag`, rather
than something generic. A restricted lab network tends to answer with a mirror
carrying a subset of PyPI, and a mirror like that satisfies a generic request
and still breaks every lab. It installs into a temporary directory that is
deleted immediately, and skips the dependency tree, because the question here is
whether this host reaches PyPI at all. Each lab resolves its own tree when you
get to it.


In [ ]:
from workshop_lab import PyPIReachable

PyPIReachable(lab).run()


## Step 14: Delete everything and confirm the account is empty

This step asks AWS what is in the account rather than replaying a list of what
this notebook created. It runs the deletes the steps above registered, then
enumerates every service they touch, deletes anything carrying the workshop tag
or a workshop name, and re-checks that each one is actually gone rather than
trusting the delete call's return.

**Run this cell even if a step above failed.** A partial run still leaves
resources, and they still cost money. It also stands on its own: if the kernel
restarted or you reopened the notebook, run steps 0, 2 and 4, then this cell.

One thing is deliberately left alone: anything CloudFormation owns. Vocareum's
lab stack deletes its own resources at session end, and pulling one out from
under it makes that delete fail.

The S3 build-source bucket used to be left in place as well, on the theory that
the account shares it. It is not any more. Nothing else in a student account
creates it, and a hand cleanup found it still sitting there after everything else
was gone, so every object in it goes and then the bucket itself.

Every object, not only this run's. Objects are named after whichever session
wrote them, so deleting only this session's leaves an earlier session's behind,
and those keep the bucket non-empty, which means deleting the bucket can never
succeed. If the bucket still refuses to go once it has been emptied, it holds
older object versions that this notebook is not permitted to list, and the cell
says so and tells you to empty it in the S3 console.

The per-endpoint AgentCore log groups under `/aws/bedrock-agentcore/runtimes/` go
too, because deleting a runtime does not delete its log group. They are deleted
after the runtimes, since a runtime keeps writing to its log group for a minute
or two after its delete call returns.

The ECR repository is deleted with `force`, which removes the built image along
with it.

In [ ]:
# The 864 lines that used to be here are in workshop_lab.teardown. That module
# carries the reasoning: why this enumerates the account rather than replaying
# the deletes the steps above registered, how a resource is recognised as this
# workshop's, why anything CloudFormation owns is left alone, and why it deletes
# in the order it does.
from workshop_lab import Teardown

Teardown(lab).run()

## Step 15: Summary

One line per check, in the order they ran. `SKIP` means a check could not run
because something it depended on failed earlier, so fix the first `FAIL` and
re-run from there.

In [ ]:
lab.summary(extra=[neo4j.result()])

## Troubleshooting

| What you see | What it usually means | What to do |
| --- | --- | --- |
| `These credential fields are empty` | The lab session ended, or never fully started | Close this tab, start the lab again, wait for the console to open, then reopen the notebook |
| The account number is not the one you were given | boto3 fell back to credentials from somewhere else | Stop. Do not trust any result below it. Restart the lab and reopen the notebook |
| Step 2 raises on the region | The lab opened somewhere this workshop cannot use | Stop and tell your instructor. The Nova embedding model is only in us-east-1 |
| Step 3 `SKIP` | The Neo4j constants still hold `REPLACE-ME` | Paste the Aura credentials you were given into the step 3 cell |
| Step 3 `Unauthorized` | Wrong Neo4j username or password | Check the values. The demos read `NEO4J_USERNAME`, not `NEO4J_USER` |
| Step 5 fails | The account cannot see AgentCore at all | Stop and tell your instructor. Nothing below step 5 can work |
| Step 8 fails pulling `ghcr.io` | CodeBuild has no outbound internet access | Report this. It blocks the workshop's deployment and you cannot fix it from the notebook |
| Step 8 `uname -m` is not `aarch64` | The build ran on the wrong architecture | Report it. AgentCore Runtime accepts ARM64 images only |
| Step 8 is denied on `privilegedMode` | The lab policy blocks Docker builds inside CodeBuild | Report this. It blocks the container build the workshop depends on |
| Step 9 reaches `READY` but the invoke returns nothing | The container started, then failed its health check | Read the Runtime's log group in CloudWatch. This is a container problem, not a permission problem |
| Step 10 `workshop tag survives a read-back` fails | AgentCore tagging is not permitted | Report it. The workshop's cleanup finds resources by tag, so untagged resources survive teardown |
| Step 11 `agentcore:ListGatewayTargets` is denied | The lab policy blocks listing a gateway's targets | Report it. Provisioning cannot find an existing target on a re-run, and cleanup cannot empty a gateway before deleting it, so gateways survive the end of the lab |
| Step 11 `lambda:AddPermission` is denied | The lab policy blocks writing a Lambda resource policy | Report it. Provisioning runs it before the first tool call and stops when it is refused |
| Step 11 `gateway tool call reaches the Lambda` fails while `agentcore:InvokeGateway` passed | The Gateway is configured but cannot invoke the Lambda | Report it. Listing tools never reaches the Lambda, so this is the check that covers the path the agent uses |
| Step 13 `PyPI is reachable from this host` fails | Outbound access to PyPI is blocked, or the index in front of it carries only a subset | Report this. Every lab in the workshop opens with a pip install, so it stops all of them at their first cell |
| Step 2b `not entitled to the model` | The account does not have access to a required chat or embedding model | Report it. Only Vocareum can move that access |
| Step 2b `IAM or an SCP refused the call` | The lab's policy does not allow the invoke action | Report it with the action name and the model id. This one is fixed in the lab, not by Vocareum |
| Step 2b `Bedrock does not know this model id` | The pinned model id has left the catalog | Report it. The ids are pinned in `workshop_lab.models` and need a new release, not a change in your account |
| Step 2b `expected one embedding` or `expected a 1024-dimension embedding` | Nova returned a response the retrieval labs cannot use | Report it with the result line and model id |
| Step 2b `Bedrock throttled the request` | Too many calls at once. Nothing was measured | Wait a minute and run the cell again |
| Step 14 leaves something `PRESENT` | A delete was refused, or is still in progress | Re-run step 14. If it persists, delete it in the AWS console. Ending the session removes the lab stack, not what this notebook created |
| Any other `AccessDenied` message | The lab policy blocks that call | Report it with the step number and the action name. Step 2b is the exception: a model refusal can be an account entitlement instead, and its own rows above say which |

## Ending your session

Run step 14 first, then end the session. Ending the session removes the lab stack
Vocareum created and nothing else. Everything this notebook created keeps running
until step 14 deletes it, and anything left running counts against the lab
budget.

## References

- [Vocareum: AWS Cloud Labs](https://help.vocareum.com/en/articles/9425419-aws-cloud-labs)
- [Vocareum: Getting Started for Students](https://help.vocareum.com/en/articles/3658973-getting-started-students)
- [AWS: AgentCore Runtime service contract](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-service-contract.html)
- [AWS: What is Amazon Bedrock AgentCore](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/what-is-bedrock-agentcore.html)
- [AWS: CodeBuild build environment reference](https://docs.aws.amazon.com/codebuild/latest/userguide/build-env-ref.html)
- [AWS: Bedrock model access](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
- [Neo4j Python driver manual](https://neo4j.com/docs/python-manual/current/)
- [boto3 documentation](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html)